
# NASA Asteroids

Cel:

```text
kilkanaście parametrów fizycznych i orbitalnych
→ wybór cech numerycznych
→ skalowanie
→ wizualizacja podobieństwa obiektów
```

> Etykieta `Hazardous` oznacza obiekt sklasyfikowany jako potencjalnie niebezpieczny. Nie oznacza to automatycznie, że planetoida znajduje się na kursie kolizyjnym z Ziemią.

## Dane

Umieść plik Kaggle/CSV lub Parquet w katalogu `data/` i ustaw `DATA_PATH`.

Notebook rozpoznaje popularne nazwy kolumn ze zbioru **NASA: Asteroids Classification**. Jeżeli pliku nie ma, tworzy syntetyczny zbiór demonstracyjny wyłącznie do sprawdzenia kodu.


In [1]:

# W środowisku lokalnym odkomentuj w razie potrzeby:
# %pip install -U polars pyarrow scikit-learn matplotlib

from __future__ import annotations

from pathlib import Path
import re

import numpy as np
import polars as pl
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler

print("Polars:", pl.__version__)


Polars: 1.42.1


In [2]:

DATA_PATH = Path("data/nasa_asteroids.csv")
CREATE_DEMO_IF_MISSING = True

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)



## 1. Zbiór demonstracyjny

Fallback zawiera przykładowe cechy przypominające parametry fizyczne i orbitalne oraz kilka brakujących wartości.


In [3]:

def create_demo_asteroids(path: Path, n_rows: int = 600) -> None:
    rng = np.random.default_rng(42)

    absolute_magnitude = rng.normal(22.0, 2.3, n_rows)
    estimated_diameter_min = np.exp(rng.normal(-2.0, 0.65, n_rows))
    estimated_diameter_max = estimated_diameter_min * rng.uniform(1.5, 2.4, n_rows)
    relative_velocity = np.exp(rng.normal(9.6, 0.35, n_rows))
    miss_distance_km = np.exp(rng.normal(15.2, 0.8, n_rows))
    minimum_orbit_intersection = np.exp(rng.normal(-3.0, 0.9, n_rows))
    eccentricity = np.clip(rng.beta(2.0, 3.5, n_rows), 0.01, 0.98)
    semi_major_axis = rng.uniform(0.7, 4.5, n_rows)
    inclination = np.clip(rng.gamma(2.0, 6.0, n_rows), 0, 70)
    orbital_period = np.exp(rng.normal(6.3, 0.45, n_rows))
    perihelion_distance = np.clip(
        semi_major_axis * (1 - eccentricity),
        0.05,
        None,
    )
    aphelion_distance = semi_major_axis * (1 + eccentricity)

    risk_score = (
        -0.85 * absolute_magnitude
        - 2.0 * np.log1p(minimum_orbit_intersection)
        + 0.15 * np.log1p(relative_velocity)
        + rng.normal(0, 0.9, n_rows)
    )
    hazardous = risk_score > np.quantile(risk_score, 0.82)

    demo = pl.DataFrame(
        {
            "Neo Reference ID": np.arange(1_000_000, 1_000_000 + n_rows),
            "Name": [f"Demo Asteroid {i}" for i in range(n_rows)],
            "Absolute Magnitude": absolute_magnitude,
            "Est Dia in KM(min)": estimated_diameter_min,
            "Est Dia in KM(max)": estimated_diameter_max,
            "Relative Velocity km per hr": relative_velocity,
            "Miss Dist.(kilometers)": miss_distance_km,
            "Minimum Orbit Intersection": minimum_orbit_intersection,
            "Eccentricity": eccentricity,
            "Semi Major Axis": semi_major_axis,
            "Inclination": inclination,
            "Orbital Period": orbital_period,
            "Perihelion Distance": perihelion_distance,
            "Aphelion Dist": aphelion_distance,
            "Hazardous": hazardous,
        }
    )

    # Kilka braków do późniejszego segmentu o imputacji.
    missing_indices = rng.choice(n_rows, size=max(1, n_rows // 25), replace=False)
    demo = demo.with_row_index("_row_id").with_columns(
        pl.when(pl.col("_row_id").is_in(missing_indices[: len(missing_indices)//2]))
        .then(None)
        .otherwise(pl.col("Absolute Magnitude"))
        .alias("Absolute Magnitude"),
        pl.when(pl.col("_row_id").is_in(missing_indices[len(missing_indices)//2:]))
        .then(None)
        .otherwise(pl.col("Minimum Orbit Intersection"))
        .alias("Minimum Orbit Intersection"),
    ).drop("_row_id")

    path.parent.mkdir(parents=True, exist_ok=True)

    if path.suffix.lower() == ".parquet":
        demo.write_parquet(path)
    else:
        demo.write_csv(path)


if not DATA_PATH.exists():
    if not CREATE_DEMO_IF_MISSING:
        raise FileNotFoundError(
            f"Nie znaleziono {DATA_PATH}. Ustaw ścieżkę do CSV lub Parquet."
        )

    create_demo_asteroids(DATA_PATH)
    print(
        f"UWAGA: utworzono syntetyczny zbiór demonstracyjny: {DATA_PATH}. "
        "Do prezentacji użyj właściwego zbioru NASA Asteroids."
    )



## 2. Wczytanie danych

Dla CSV i Parquet korzystamy z Lazy API Polars.


In [4]:

def scan_dataset(path: Path) -> pl.LazyFrame:
    suffix = path.suffix.lower()

    if suffix == ".csv":
        return pl.scan_csv(
            path,
            infer_schema_length=10_000,
            try_parse_dates=False,
        )

    if suffix == ".parquet":
        return pl.scan_parquet(path)

    raise ValueError("Obsługiwane formaty: CSV i Parquet.")


raw_lf = scan_dataset(DATA_PATH)
raw_lf.collect_schema()


Schema([('Neo Reference ID', Int64),
        ('Name', String),
        ('Absolute Magnitude', Float64),
        ('Est Dia in KM(min)', Float64),
        ('Est Dia in KM(max)', Float64),
        ('Relative Velocity km per hr', Float64),
        ('Miss Dist.(kilometers)', Float64),
        ('Minimum Orbit Intersection', Float64),
        ('Eccentricity', Float64),
        ('Semi Major Axis', Float64),
        ('Inclination', Float64),
        ('Orbital Period', Float64),
        ('Perihelion Distance', Float64),
        ('Aphelion Dist', Float64),
        ('Hazardous', Boolean)])


## 3. Ujednolicenie nazw kolumn

Różne kopie zbioru mogą nieznacznie różnić się zapisem nazw. Tworzymy nazwy w formacie `snake_case`.


In [5]:

def to_snake_case(name: str) -> str:
    name = name.strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    return name.strip("_")


source_columns = raw_lf.collect_schema().names()
rename_map = {column: to_snake_case(column) for column in source_columns}

asteroids_lf = raw_lf.rename(rename_map)
asteroids_lf.collect_schema()


Schema([('neo_reference_id', Int64),
        ('name', String),
        ('absolute_magnitude', Float64),
        ('est_dia_in_km_min', Float64),
        ('est_dia_in_km_max', Float64),
        ('relative_velocity_km_per_hr', Float64),
        ('miss_dist_kilometers', Float64),
        ('minimum_orbit_intersection', Float64),
        ('eccentricity', Float64),
        ('semi_major_axis', Float64),
        ('inclination', Float64),
        ('orbital_period', Float64),
        ('perihelion_distance', Float64),
        ('aphelion_dist', Float64),
        ('hazardous', Boolean)])

In [6]:

PREFERRED_NUMERIC_FEATURES = [
    "absolute_magnitude",
    "est_dia_in_km_min",
    "est_dia_in_km_max",
    "relative_velocity_km_per_hr",
    "miss_dist_kilometers",
    "minimum_orbit_intersection",
    "jupiter_tisserand_invariant",
    "orbit_uncertainity",
    "eccentricity",
    "semi_major_axis",
    "inclination",
    "asc_node_longitude",
    "orbital_period",
    "perihelion_distance",
    "perihelion_arg",
    "aphelion_dist",
    "mean_anomaly",
    "mean_motion",
]

schema = asteroids_lf.collect_schema()
available_columns = set(schema.names())

selected_features = [
    column
    for column in PREFERRED_NUMERIC_FEATURES
    if column in available_columns
]

if len(selected_features) < 3:
    raise ValueError(
        "Znaleziono zbyt mało rozpoznanych cech numerycznych. "
        f"Dostępne kolumny: {schema.names()}"
    )

print("Wybrane cechy:")
for feature in selected_features:
    print("-", feature)


Wybrane cechy:
- absolute_magnitude
- est_dia_in_km_min
- est_dia_in_km_max
- relative_velocity_km_per_hr
- miss_dist_kilometers
- minimum_orbit_intersection
- eccentricity
- semi_major_axis
- inclination
- orbital_period
- perihelion_distance
- aphelion_dist



## 4. Konwersja typów i analiza braków

W tej części nie omawiamy jeszcze imputacji. Sprawdzamy jedynie kompletność cech.


In [7]:

typed_lf = asteroids_lf.with_columns(
    [
        pl.col(column).cast(pl.Float64, strict=False)
        for column in selected_features
    ]
)

missing_report = typed_lf.select(
    [
        pl.col(column).is_null().sum().alias(column)
        for column in selected_features
    ]
).collect()

missing_report


absolute_magnitude,est_dia_in_km_min,est_dia_in_km_max,relative_velocity_km_per_hr,miss_dist_kilometers,minimum_orbit_intersection,eccentricity,semi_major_axis,inclination,orbital_period,perihelion_distance,aphelion_dist
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
12,0,0,0,0,12,0,0,0,0,0,0


In [8]:

complete_cases = (
    typed_lf
    .select(selected_features)
    .drop_nulls()
    .collect()
)

print(f"Kompletne rekordy: {complete_cases.height:,}")
print(f"Liczba cech: {len(selected_features)}")


Kompletne rekordy: 576
Liczba cech: 12



## 5. Skalowanie

Bez skalowania cechy zapisane w dużych jednostkach, np. odległość w kilometrach, mogłyby zdominować komponenty.


In [9]:

X_asteroids = complete_cases.to_numpy()

if not np.isfinite(X_asteroids).all():
    raise ValueError("Dane zawierają wartości nieskończone.")

asteroid_scaler = StandardScaler()
X_asteroids_scaled = asteroid_scaler.fit_transform(X_asteroids)

pl.DataFrame(
    {
        "feature": selected_features,
        "mean_after_scaling": X_asteroids_scaled.mean(axis=0),
        "std_after_scaling": X_asteroids_scaled.std(axis=0),
    }
)


feature,mean_after_scaling,std_after_scaling
str,f64,f64
"""absolute_magnitude""",2.3261e-15,1.0
"""est_dia_in_km_min""",-2.1588e-17,1.0
"""est_dia_in_km_max""",1.9737e-16,1.0
"""relative_velocity_km_per_hr""",2.4672e-16,1.0
"""miss_dist_kilometers""",1.6962e-17,1.0
…,…,…
"""semi_major_axis""",2.6676e-16,1.0
"""inclination""",-1.4186e-16,1.0
"""orbital_period""",-9.5603e-17,1.0


## Brakujące dane i wykrywanie anomalii

Przepływ:

```text
dane z brakami
→ porównanie SimpleImputer i KNNImputer
→ dopasowanie preprocessingu na zbiorze treningowym
→ przygotowanie kompletnej macierzy
→ Isolation Forest / LOF / K-Means / DBSCAN
→ porównanie wykrytych anomalii
```

W tej części rozróżniamy:

- **brak wartości** — nie znamy pomiaru,
- **anomalię** — wartość istnieje, ale rekord odbiega od typowego wzorca,
- **błąd danych** — rekord może być niepoprawny, ale algorytm sam tego nie rozstrzyga.



## 6. Przygotowanie macierzy z brakami

Korzystamy ze wszystkich rekordów, również tych zawierających brakujące wartości.

Aby porównać metody imputacji, dodatkowo ukrywamy niewielką część znanych wartości. Dzięki temu po imputacji możemy porównać wynik z rzeczywistą wartością.

Sztuczne braki są tworzone wyłącznie w kopii danych na potrzeby demonstracji.


In [10]:

from sklearn.cluster import DBSCAN, KMeans
from sklearn.ensemble import IsolationForest
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import LocalOutlierFactor, NearestNeighbors
from sklearn.pipeline import Pipeline

asteroid_features_df = typed_lf.select(selected_features).collect()
X_original = asteroid_features_df.to_numpy()

if X_original.shape[0] < 30:
    raise ValueError(
        "Do demonstracji potrzebujemy co najmniej 30 rekordów."
    )

real_missing_count = int(np.isnan(X_original).sum())
print("Rzeczywiste braki:", real_missing_count)
print("Kształt macierzy:", X_original.shape)


Rzeczywiste braki: 24
Kształt macierzy: (600, 12)


In [11]:

rng = np.random.default_rng(42)

X_demo_missing = X_original.copy()
observed_positions = np.argwhere(np.isfinite(X_original))

SIMULATED_MISSING_RATE = 0.03
n_to_mask = max(
    1,
    int(len(observed_positions) * SIMULATED_MISSING_RATE),
)

chosen_positions = observed_positions[
    rng.choice(
        len(observed_positions),
        size=n_to_mask,
        replace=False,
    )
]

synthetic_missing_mask = np.zeros_like(
    X_original,
    dtype=bool,
)
synthetic_missing_mask[
    chosen_positions[:, 0],
    chosen_positions[:, 1],
] = True

X_demo_missing[synthetic_missing_mask] = np.nan

print("Dodane braki demonstracyjne:", n_to_mask)
print(
    "Łączna liczba braków w kopii:",
    int(np.isnan(X_demo_missing).sum()),
)


Dodane braki demonstracyjne: 215
Łączna liczba braków w kopii: 239



## 7. SimpleImputer

`SimpleImputer` uzupełnia każdą kolumnę niezależnie.

Dla zmiennych numerycznych typowe strategie to:

- `mean` — średnia,
- `median` — mediana,
- `constant` — stała wartość.

Dla zmiennych kategorycznych często wykorzystuje się:

- `most_frequent` — najczęstsza wartość,
- `constant`, np. `"unknown"`.

Mediana jest zwykle mniej wrażliwa na skrajne wartości niż średnia.


In [12]:

row_indices = np.arange(X_demo_missing.shape[0])

train_indices, test_indices = train_test_split(
    row_indices,
    test_size=0.25,
    random_state=42,
)

X_train_missing = X_demo_missing[train_indices]
X_test_missing = X_demo_missing[test_indices]

simple_imputer = SimpleImputer(
    strategy="median",
    add_indicator=False,
)

simple_imputer.fit(X_train_missing)

X_train_simple = simple_imputer.transform(
    X_train_missing
)
X_test_simple = simple_imputer.transform(
    X_test_missing
)

print("Braki po imputacji train:", np.isnan(X_train_simple).sum())
print("Braki po imputacji test:", np.isnan(X_test_simple).sum())


Braki po imputacji train: 0
Braki po imputacji test: 0



## 8. KNNImputer

`KNNImputer` szuka rekordów podobnych według dostępnych cech i wykorzystuje ich wartości do uzupełnienia braku.

Ponieważ metoda opiera się na odległościach, wcześniej standaryzujemy kolumny. `StandardScaler` jest dopasowywany wyłącznie na zbiorze treningowym, a następnie stosowany do zbioru testowego.

Wagi `distance` powodują, że bliżsi sąsiedzi mają większy wpływ na wynik.


In [13]:

knn_scaler = StandardScaler()
knn_scaler.fit(X_train_missing)

X_train_scaled_with_nan = knn_scaler.transform(
    X_train_missing
)
X_test_scaled_with_nan = knn_scaler.transform(
    X_test_missing
)

knn_imputer = KNNImputer(
    n_neighbors=5,
    weights="distance",
)

knn_imputer.fit(X_train_scaled_with_nan)

X_train_knn_scaled = knn_imputer.transform(
    X_train_scaled_with_nan
)
X_test_knn_scaled = knn_imputer.transform(
    X_test_scaled_with_nan
)

# Powrót do oryginalnych jednostek tylko do porównania jakości imputacji.
X_train_knn = knn_scaler.inverse_transform(
    X_train_knn_scaled
)
X_test_knn = knn_scaler.inverse_transform(
    X_test_knn_scaled
)

print("Braki po KNN train:", np.isnan(X_train_knn).sum())
print("Braki po KNN test:", np.isnan(X_test_knn).sum())


Braki po KNN train: 0
Braki po KNN test: 0



## 9. Porównanie imputacji na sztucznie ukrytych wartościach

Porównujemy tylko te komórki w zbiorze testowym, które:

1. pierwotnie miały znaną wartość,
2. zostały przez nas celowo ukryte.

Ze względu na różne jednostki cech obliczamy błąd znormalizowany przez odchylenie standardowe danej kolumny.


In [14]:

test_synthetic_mask = synthetic_missing_mask[test_indices]
X_test_truth = X_original[test_indices]

feature_scale = np.nanstd(
    X_original[train_indices],
    axis=0,
)
feature_scale = np.where(
    feature_scale > 0,
    feature_scale,
    1.0,
)

def normalized_imputation_mae(
    imputed: np.ndarray,
    truth: np.ndarray,
    mask: np.ndarray,
    scale: np.ndarray,
) -> float:
    row_idx, col_idx = np.where(mask)

    if len(row_idx) == 0:
        return float("nan")

    normalized_errors = (
        np.abs(
            imputed[row_idx, col_idx]
            - truth[row_idx, col_idx]
        )
        / scale[col_idx]
    )
    return float(np.mean(normalized_errors))


simple_error = normalized_imputation_mae(
    X_test_simple,
    X_test_truth,
    test_synthetic_mask,
    feature_scale,
)

knn_error = normalized_imputation_mae(
    X_test_knn,
    X_test_truth,
    test_synthetic_mask,
    feature_scale,
)

imputation_comparison = pl.DataFrame(
    {
        "method": [
            "SimpleImputer - median",
            "KNNImputer - 5 neighbors",
        ],
        "normalized_mae": [
            simple_error,
            knn_error,
        ],
    }
).sort("normalized_mae")

imputation_comparison


method,normalized_mae
str,f64
"""KNNImputer - 5 neighbors""",0.494365
"""SimpleImputer - median""",0.663408



Niższy błąd w tej pojedynczej symulacji nie dowodzi, że dana metoda zawsze będzie najlepsza.

Wynik zależy między innymi od:

- mechanizmu powstawania braków,
- liczby rekordów,
- zależności pomiędzy cechami,
- skalowania,
- liczby sąsiadów,
- obecności anomalii.

Prosta mediana może być równie dobra albo lepsza od bardziej złożonej metody.



## 10. Dlaczego imputer dopasowujemy tylko na danych treningowych?

Imputer uczy się informacji ze zbioru:

- median,
- średnich,
- układu najbliższych sąsiadów,
- zależności pomiędzy kolumnami.

Jeżeli dopasujemy go na całych danych przed oceną modelu, informacje ze zbioru testowego wpłyną na preprocessing. Jest to data leakage.

Poprawny schemat:

```text
train → fit imputer/scaler
test  → tylko transform
```

W następnym punkcie połączymy te operacje z modelem w `Pipeline`.



# Wykrywanie anomalii

Dla historycznej, eksploracyjnej analizy przygotowujemy kompletną i przeskalowaną macierz wszystkich asteroid.

Tutaj preprocessing jest dopasowany do badanego zbioru referencyjnego. W systemie obsługującym nowe obiekty należałoby zachować imputer, scaler i detektor dopasowane na danych referencyjnych, a nowe dane wyłącznie transformować i oceniać.


In [15]:

anomaly_preprocessor = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "imputer",
            KNNImputer(
                n_neighbors=5,
                weights="distance",
            ),
        ),
    ]
)

X_anomaly = anomaly_preprocessor.fit_transform(
    X_original
)

if not np.isfinite(X_anomaly).all():
    raise ValueError(
        "Po preprocessingu pozostały wartości NaN lub nieskończone."
    )

print("Macierz do anomaly detection:", X_anomaly.shape)


Macierz do anomaly detection: (600, 12)



## 11. Isolation Forest

Isolation Forest tworzy wiele losowych drzew i próbuje izolować obserwacje przez losowe podziały.

Nietypowy rekord zwykle wymaga mniejszej liczby podziałów, ponieważ znajduje się daleko od gęstej części danych.

Parametr `contamination` określa oczekiwany udział rekordów, które zostaną oznaczone jako anomalie. Nie jest to wartość, którą powinniśmy ustawiać bez uzasadnienia biznesowego.


In [16]:

EXPECTED_ANOMALY_SHARE = 0.03

isolation_forest = IsolationForest(
    n_estimators=300,
    contamination=EXPECTED_ANOMALY_SHARE,
    random_state=42,
)

isolation_labels = isolation_forest.fit_predict(
    X_anomaly
)

# Wyższy wynik = bardziej nietypowy rekord.
isolation_anomaly_score = -isolation_forest.score_samples(
    X_anomaly
)

print(
    "Isolation Forest — anomalie:",
    int(np.sum(isolation_labels == -1)),
)


Isolation Forest — anomalie: 18



## 12. Local Outlier Factor

LOF porównuje lokalną gęstość rekordu z gęstością jego najbliższych sąsiadów.

Rekord może zostać uznany za nietypowy, jeżeli znajduje się w znacznie rzadszym obszarze niż jego otoczenie.

LOF może więc znajdować lokalne anomalie, które nie muszą być ekstremalne względem całego zbioru.


In [17]:

lof_neighbors = min(
    20,
    max(2, X_anomaly.shape[0] - 1),
)

lof = LocalOutlierFactor(
    n_neighbors=lof_neighbors,
    contamination=EXPECTED_ANOMALY_SHARE,
)

lof_labels = lof.fit_predict(X_anomaly)

# negative_outlier_factor_ jest bardziej ujemny dla anomalii.
lof_anomaly_score = -lof.negative_outlier_factor_

print(
    "LOF — anomalie:",
    int(np.sum(lof_labels == -1)),
)


LOF — anomalie: 18



## 13. Odległość od centroidu K-Means

To podejście mierzy nietypowość względem najbliższego segmentu.

Po dopasowaniu K-Means obliczamy odległość każdego obiektu od przypisanego centroidu. Za potencjalne anomalie uznajemy rekordy powyżej wybranego kwantyla odległości.

Jest to heurystyka podobna do tej zastosowanej wcześniej dla klientów Online Retail.


In [18]:

ANOMALY_K = min(
    5,
    max(2, X_anomaly.shape[0] // 20),
)

asteroid_kmeans = KMeans(
    n_clusters=ANOMALY_K,
    n_init=20,
    random_state=42,
)

asteroid_cluster_labels = asteroid_kmeans.fit_predict(
    X_anomaly
)

asteroid_centroid_distances = asteroid_kmeans.transform(
    X_anomaly
)

assigned_centroid_distance = asteroid_centroid_distances[
    np.arange(len(asteroid_cluster_labels)),
    asteroid_cluster_labels,
]

centroid_threshold = float(
    np.quantile(
        assigned_centroid_distance,
        1 - EXPECTED_ANOMALY_SHARE,
    )
)

centroid_anomaly_labels = np.where(
    assigned_centroid_distance >= centroid_threshold,
    -1,
    1,
)

print(
    "Odległość od centroidu — anomalie:",
    int(np.sum(centroid_anomaly_labels == -1)),
)


Odległość od centroidu — anomalie: 18



## 14. DBSCAN i rekordy `-1`

DBSCAN nie jest klasycznym detektorem anomalii, ale rekordy nieprzypisane do żadnego odpowiednio gęstego obszaru otrzymują etykietę `-1`.

Dobór `eps` ma bardzo duży wpływ na udział szumu. Poniżej stosujemy prostą heurystykę opartą na odległości do dziesiątego sąsiada.


In [19]:

dbscan_min_samples = min(
    10,
    max(3, X_anomaly.shape[0] // 50),
)

neighbors = NearestNeighbors(
    n_neighbors=dbscan_min_samples,
)
neighbors.fit(X_anomaly)

neighbor_distances, _ = neighbors.kneighbors(
    X_anomaly
)

k_distance = neighbor_distances[:, -1]
dbscan_eps = float(np.quantile(k_distance, 0.95))

asteroid_dbscan = DBSCAN(
    eps=dbscan_eps,
    min_samples=dbscan_min_samples,
)

dbscan_labels = asteroid_dbscan.fit_predict(
    X_anomaly
)

print("DBSCAN eps:", round(dbscan_eps, 3))
print(
    "DBSCAN — noise:",
    int(np.sum(dbscan_labels == -1)),
)
print(
    "DBSCAN — zwykłe klastry:",
    len(set(dbscan_labels) - {-1}),
)


DBSCAN eps: 3.608
DBSCAN — noise: 9
DBSCAN — zwykłe klastry: 1



## 15. Porównanie wyników

Nie oczekujemy, że wszystkie metody wskażą dokładnie te same rekordy.

- Isolation Forest szuka obserwacji łatwych do odizolowania globalnie.
- LOF porównuje lokalną gęstość.
- K-Means mierzy odległość od centrum najbliższej grupy.
- DBSCAN wskazuje rekordy poza gęstymi obszarami.

Zgodność kilku metod może zwiększać priorytet ręcznej weryfikacji, ale nie dowodzi automatycznie, że rekord jest błędem.


In [20]:

anomaly_flags = pl.DataFrame(
    {
        "isolation_forest_anomaly": isolation_labels == -1,
        "lof_anomaly": lof_labels == -1,
        "centroid_anomaly": centroid_anomaly_labels == -1,
        "dbscan_noise": dbscan_labels == -1,
        "isolation_score": isolation_anomaly_score,
        "lof_score": lof_anomaly_score,
        "centroid_distance": assigned_centroid_distance,
        "dbscan_cluster_id": dbscan_labels,
    }
).with_columns(
    pl.sum_horizontal(
        pl.col(
            "isolation_forest_anomaly",
            "lof_anomaly",
            "centroid_anomaly",
            "dbscan_noise",
        ).cast(pl.Int8)
    ).alias("number_of_methods_flagging")
)

method_comparison = anomaly_flags.select(
    pl.len().alias("records"),
    pl.col("isolation_forest_anomaly").sum().alias(
        "isolation_forest"
    ),
    pl.col("lof_anomaly").sum().alias("lof"),
    pl.col("centroid_anomaly").sum().alias(
        "centroid_distance"
    ),
    pl.col("dbscan_noise").sum().alias("dbscan_noise"),
    (
        pl.col("number_of_methods_flagging") >= 2
    ).sum().alias("flagged_by_at_least_two"),
)

method_comparison


records,isolation_forest,lof,centroid_distance,dbscan_noise,flagged_by_at_least_two
u32,u32,u32,u32,u32,u32
600,18,18,18,9,17


## 16. Tabela najbardziej nietypowych obiektów

Do tabeli wynikowej dołączamy identyfikator lub nazwę obiektu, jeżeli są dostępne.


In [21]:

all_asteroids_df = asteroids_lf.collect()

identifier_candidates = [
    "neo_reference_id",
    "name",
    "id",
]

identifier_columns = [
    column
    for column in identifier_candidates
    if column in all_asteroids_df.columns
]

result_base_columns = [
    *identifier_columns,
    *selected_features,
]

asteroid_anomaly_results = (
    all_asteroids_df
    .select(result_base_columns)
    .hstack(anomaly_flags)
)

asteroid_anomaly_results.sort(
    [
        "number_of_methods_flagging",
        "isolation_score",
    ],
    descending=[True, True],
).head(20)


neo_reference_id,name,absolute_magnitude,est_dia_in_km_min,est_dia_in_km_max,relative_velocity_km_per_hr,miss_dist_kilometers,minimum_orbit_intersection,eccentricity,semi_major_axis,inclination,orbital_period,perihelion_distance,aphelion_dist,isolation_forest_anomaly,lof_anomaly,centroid_anomaly,dbscan_noise,isolation_score,lof_score,centroid_distance,dbscan_cluster_id,number_of_methods_flagging
i64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,bool,bool,f64,f64,f64,i64,i8
1000477,"""Demo Asteroid 477""",19.373542,0.840286,1.928605,7737.098659,4.2045e6,0.178882,0.763632,1.134703,13.218812,402.922507,0.268208,2.001198,true,true,true,true,0.621059,1.832391,7.216665,-1,4
1000246,"""Demo Asteroid 246""",19.477393,1.068499,2.172965,20926.973618,3.3647e6,0.097019,0.201766,4.048428,30.78097,384.607211,3.231593,4.865263,true,true,true,true,0.613873,2.241997,8.846511,-1,4
1000150,"""Demo Asteroid 150""",21.63514,0.136346,0.291129,15263.038508,5.5327e6,0.762301,0.263927,0.83448,39.311244,256.280913,0.614238,1.054723,true,true,true,true,0.584086,2.152539,6.157015,-1,4
1000031,"""Demo Asteroid 31""",21.065245,0.709683,1.689593,22642.977375,1.3796e7,0.071485,0.163013,1.068715,8.95225,294.75095,0.894501,1.24293,true,true,true,true,0.579508,1.610159,5.84835,-1,4
1000318,"""Demo Asteroid 318""",24.276043,0.328318,0.562212,15898.583024,3.6466e6,0.03339,0.681555,3.884323,8.612302,3332.975806,1.236942,6.531703,true,true,true,true,0.554039,2.550109,9.441842,-1,4
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1000499,"""Demo Asteroid 499""",18.56136,0.303393,0.658806,17263.185898,2.0760e6,0.01672,0.475908,2.326072,2.424014,1880.155466,1.219077,3.433068,false,true,true,false,0.50623,1.514481,5.062327,0,2
1000375,"""Demo Asteroid 375""",27.353602,0.068767,0.151164,39514.66852,929981.271226,0.05115,0.280834,2.928262,11.432698,664.974693,2.105905,3.750619,false,true,true,false,0.492634,1.596846,5.187447,0,2
1000453,"""Demo Asteroid 453""",26.777055,0.663209,1.230645,17541.134055,2.3463e6,0.134715,0.207481,3.578935,11.580136,705.768733,2.836372,4.321497,true,false,false,false,0.551055,1.400348,4.31197,0,1


## Klasyfikacja i podejmowanie decyzji

Problem analityczny:

> Czy na podstawie parametrów fizycznych i orbitalnych można zaklasyfikować asteroidę jako potencjalnie niebezpieczną?

Przepływ:

```text
dane i target
→ podział train/test
→ ColumnTransformer
   ├── cechy numeryczne: KNNImputer → StandardScaler
   └── cechy kategoryczne: SimpleImputer → OneHotEncoder
→ Pipeline
→ klasyfikator
→ predict / predict_proba
→ confusion matrix i metryki
→ próg decyzji
```

Porównamy:

- model bazowy `DummyClassifier`,
- `LogisticRegression`,
- `RandomForestClassifier`.

> `Hazardous` oznacza klasę potentially hazardous asteroid, a nie prognozę konkretnego zderzenia z Ziemią.


## 17. Importy do klasyfikacji

In [22]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

print("Pandas:", pd.__version__)


Pandas: 3.0.3


## 18. Przygotowanie zmiennej docelowej

W klasyfikacji potrzebujemy etykiety, którą model ma przewidywać. Nazywamy ją:

- targetem,
- zmienną docelową,
- etykietą klasy.

W popularnym zbiorze NASA Asteroids kolumna nazywa się zwykle `Hazardous`. Po ujednoliceniu nazw oczekujemy `hazardous`.


In [23]:

classification_df = asteroids_lf.collect().to_pandas()

TARGET_CANDIDATES = [
    "hazardous",
    "is_hazardous",
    "potentially_hazardous",
]

target_column = next(
    (
        column
        for column in TARGET_CANDIDATES
        if column in classification_df.columns
    ),
    None,
)

if target_column is None:
    raise ValueError(
        "Nie znaleziono targetu Hazardous. "
        f"Dostępne kolumny: {classification_df.columns.tolist()}"
    )


def normalize_binary_target(series: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.astype(int)

    if pd.api.types.is_numeric_dtype(series):
        numeric = pd.to_numeric(series, errors="coerce")
        unique = set(numeric.dropna().unique().tolist())

        if unique.issubset({0, 1}):
            return numeric.astype("Int64")

        return (numeric > 0).astype("Int64")

    normalized = (
        series.astype("string")
        .str.strip()
        .str.lower()
    )

    mapping = {
        "true": 1,
        "yes": 1,
        "y": 1,
        "1": 1,
        "hazardous": 1,
        "false": 0,
        "no": 0,
        "n": 0,
        "0": 0,
        "not hazardous": 0,
        "non-hazardous": 0,
    }

    return normalized.map(mapping).astype("Int64")


classification_df["target"] = normalize_binary_target(
    classification_df[target_column]
)

classification_df = classification_df.dropna(
    subset=["target"]
).copy()

classification_df["target"] = (
    classification_df["target"].astype(int)
)

target_distribution = (
    classification_df["target"]
    .value_counts()
    .rename_axis("class")
    .reset_index(name="records")
    .sort_values("class")
)

target_distribution["share"] = (
    target_distribution["records"]
    / target_distribution["records"].sum()
)

target_distribution


,class,records,share
0,0,492,0.82
1,1,108,0.18



### Niezbalansowane klasy

W zbiorach dotyczących ryzyka klasa pozytywna jest często rzadsza.

Dlatego sama accuracy może być myląca. Model przewidujący zawsze klasę większościową może uzyskać wysoką accuracy, ale nie wykryć ani jednego potencjalnie niebezpiecznego obiektu.

Będziemy analizować także:

- precision,
- recall,
- F1,
- balanced accuracy,
- ROC AUC,
- average precision.


## 19. Cechy numeryczne i kategoryczne

Zbiór NASA jest głównie numeryczny. `ColumnTransformer` pokażemy jednak w wersji obsługującej oba typy danych.

Najpierw szukamy użytecznej kolumny kategorycznej. Jeżeli jej nie ma, tworzymy interpretowalną kategorię `inclination_band` na podstawie nachylenia orbity i usuwamy źródłową kolumnę `inclination` z cech numerycznych, aby nie przekazywać tej samej informacji dwa razy.


In [24]:

numeric_features = [
    feature
    for feature in selected_features
    if feature in classification_df.columns
]

categorical_candidates = [
    "orbiting_body",
    "equinox",
    "orbit_class_type",
    "orbit_class_description",
]

categorical_features = [
    column
    for column in categorical_candidates
    if (
        column in classification_df.columns
        and classification_df[column].nunique(dropna=True) > 1
        and classification_df[column].nunique(dropna=True) <= 30
    )
]

engineered_category = None

if not categorical_features:
    if "inclination" in classification_df.columns:
        engineered_category = "inclination_band"
        classification_df[engineered_category] = pd.cut(
            pd.to_numeric(
                classification_df["inclination"],
                errors="coerce",
            ),
            bins=[-np.inf, 10, 25, np.inf],
            labels=["low", "medium", "high"],
        )
        categorical_features = [engineered_category]
        numeric_features = [
            feature
            for feature in numeric_features
            if feature != "inclination"
        ]

    elif "eccentricity" in classification_df.columns:
        engineered_category = "eccentricity_band"
        classification_df[engineered_category] = pd.cut(
            pd.to_numeric(
                classification_df["eccentricity"],
                errors="coerce",
            ),
            bins=[-np.inf, 0.3, 0.6, np.inf],
            labels=["low", "medium", "high"],
        )
        categorical_features = [engineered_category]
        numeric_features = [
            feature
            for feature in numeric_features
            if feature != "eccentricity"
        ]

if not numeric_features:
    raise ValueError("Nie znaleziono cech numerycznych do modelu.")

print("Cechy numeryczne:")
for feature in numeric_features:
    print("-", feature)

print("\nCechy kategoryczne:")
for feature in categorical_features:
    print("-", feature)

if engineered_category:
    print(
        "\nUtworzono demonstracyjną cechę:",
        engineered_category,
    )


Cechy numeryczne:
- absolute_magnitude
- est_dia_in_km_min
- est_dia_in_km_max
- relative_velocity_km_per_hr
- miss_dist_kilometers
- minimum_orbit_intersection
- eccentricity
- semi_major_axis
- orbital_period
- perihelion_distance
- aphelion_dist

Cechy kategoryczne:
- inclination_band

Utworzono demonstracyjną cechę: inclination_band



## 20. Podział na dane treningowe i testowe

Zbiór treningowy służy do:

- obliczenia parametrów imputacji,
- dopasowania skalera,
- poznania kategorii dla OneHotEncoder,
- wyuczenia modelu.

Zbiór testowy pozostaje niewidoczny podczas uczenia i służy do końcowej oceny.

Używamy `stratify=y`, aby zachować podobny udział klasy pozytywnej w obu częściach.


In [25]:

all_input_features = [
    *numeric_features,
    *categorical_features,
]

X = classification_df[all_input_features].copy()
y = classification_df["target"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

split_summary = pd.DataFrame(
    {
        "dataset": ["train", "test"],
        "records": [len(y_train), len(y_test)],
        "positive_share": [
            y_train.mean(),
            y_test.mean(),
        ],
    }
)

split_summary


,dataset,records,positive_share
0,train,450,0.18
1,test,150,0.18



## 21. ColumnTransformer

`ColumnTransformer` stosuje różne operacje do różnych grup kolumn.

### Cechy numeryczne

```text
KNNImputer
→ StandardScaler
```

### Cechy kategoryczne

```text
SimpleImputer(strategy="most_frequent")
→ OneHotEncoder(handle_unknown="ignore")
```

W tym przykładzie zachowujemy kolejność KNNImputer → StandardScaler, aby pokazać typowy, czytelny pipeline. Trzeba jednak pamiętać, że KNNImputer korzysta z odległości, więc przy bardzo różnych jednostkach warto rozważyć wcześniejszą normalizację lub bardziej wyspecjalizowany preprocessing.


In [26]:

def build_preprocessor(
    numeric_columns: list[str],
    categorical_columns: list[str],
) -> ColumnTransformer:
    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                KNNImputer(
                    n_neighbors=5,
                    weights="distance",
                ),
            ),
            (
                "scaler",
                StandardScaler(),
            ),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent",
                ),
            ),
            (
                "one_hot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
            ),
        ]
    )

    transformers = [
        (
            "numeric",
            numeric_pipeline,
            numeric_columns,
        ),
    ]

    if categorical_columns:
        transformers.append(
            (
                "categorical",
                categorical_pipeline,
                categorical_columns,
            )
        )

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        verbose_feature_names_out=False,
    )


preprocessor = build_preprocessor(
    numeric_features,
    categorical_features,
)

preprocessor


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{feature_name}__{transformer_name}""``. See :meth:`str.format` method from the standard library for more info... versionadded:: 1.0.. versionchanged:: 1.6 `verbose_feature_names_out` can be a callable or a string to be formatted.",False
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per tran


## 22. Model bazowy

Zanim ocenimy właściwe modele, tworzymy baseline.

`DummyClassifier(strategy="prior")` ignoruje cechy i przewiduje klasy zgodnie z rozkładem klas w danych treningowych.

Model ML powinien dostarczyć wyraźnie lepszy wynik niż taki prosty punkt odniesienia.


In [27]:

baseline_pipeline = Pipeline(
    steps=[
        (
            "preprocessing",
            build_preprocessor(
                numeric_features,
                categorical_features,
            ),
        ),
        (
            "classifier",
            DummyClassifier(
                strategy="prior",
                random_state=42,
            ),
        ),
    ]
)

baseline_pipeline.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessing', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](12,)","['absolute_magnitude','est_dia_in_km_min','est_dia_in_km_max',..., 'perihelion_distance','aphelion_dist','inclination_band']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,12
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the 


## 23. Logistic Regression i Random Forest

### Logistic Regression

- model liniowy,
- szybki i stosunkowo łatwy do interpretacji,
- zwraca prawdopodobieństwo klasy,
- korzysta ze skalowania,
- może być dobrym baseline'em predykcyjnym.

### Random Forest

- zespół wielu drzew decyzyjnych,
- potrafi modelować nieliniowe zależności i interakcje,
- zwykle nie wymaga skalowania, ale może korzystać ze wspólnego pipeline'u,
- również udostępnia `predict_proba`.

Ustawiamy `class_weight="balanced"`, aby zwiększyć wagę rzadszej klasy pozytywnej.


In [28]:

logistic_pipeline = Pipeline(
    steps=[
        (
            "preprocessing",
            build_preprocessor(
                numeric_features,
                categorical_features,
            ),
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=2_000,
                class_weight="balanced",
                random_state=42,
            ),
        ),
    ]
)

random_forest_pipeline = Pipeline(
    steps=[
        (
            "preprocessing",
            build_preprocessor(
                numeric_features,
                categorical_features,
            ),
        ),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=400,
                min_samples_leaf=2,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

logistic_pipeline.fit(X_train, y_train)
random_forest_pipeline.fit(X_train, y_train)

print("Modele zostały dopasowane.")


Modele zostały dopasowane.



## 24. Funkcja oceny modeli

`predict()` zwraca decyzję klasową.

`predict_proba()` zwraca prawdopodobieństwo klasy pozytywnej. Pozwala ono:

- tworzyć ranking obiektów,
- zmieniać próg decyzji,
- dostosować model do kosztu różnych błędów.

Prawdopodobieństwa modelu nie zawsze są idealnie skalibrowane. W systemie wysokiego ryzyka należałoby dodatkowo ocenić kalibrację.


In [29]:
def evaluate_classifier(
    name: str,
    model: Pipeline,
    X_eval: pd.DataFrame,
    y_eval: pd.Series,
) -> dict[str, float | str]:
    predictions = model.predict(X_eval)
    probabilities = model.predict_proba(X_eval)[:, 1]

    return {
        "model": name,
        "accuracy": accuracy_score(
            y_eval,
            predictions,
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_eval,
            predictions,
        ),
        "precision": precision_score(
            y_eval,
            predictions,
            zero_division=0,
        ),
        "recall": recall_score(
            y_eval,
            predictions,
            zero_division=0,
        ),
        "f1": f1_score(
            y_eval,
            predictions,
            zero_division=0,
        ),
        "roc_auc": roc_auc_score(
            y_eval,
            probabilities,
        ),
        "average_precision": average_precision_score(
            y_eval,
            probabilities,
        ),
    }


models = {
    "DummyClassifier": baseline_pipeline,
    "LogisticRegression": logistic_pipeline,
    "RandomForest": random_forest_pipeline,
}

evaluation_results = pd.DataFrame(
    [
        evaluate_classifier(
            name,
            model,
            X_test,
            y_test,
        )
        for name, model in models.items()
    ]
).sort_values(
    "average_precision",
    ascending=False,
)

evaluation_results


,model,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,average_precision
2,RandomForest,0.886667,0.930894,0.613636,1.000000,0.760563,0.957844,0.830198
1,LogisticRegression,0.826667,0.879855,0.509804,0.962963,0.666667,0.944595,0.780790
0,DummyClassifier,0.820000,0.500000,0.000000,0.000000,0.000000,0.500000,0.180000
